# 04. 멀티 Tool Agent 통합 실습


**통합 Tool 4개:** `get_city_time_tz` · `get_us_stock_price` · `summarize_pdf_document` · `search_university_rules`

---
## 0. 설치

In [1]:
!pip install openai python-dotenv pymupdf pytz yfinance

---
## 1. OpenAI API 연결 (3.2)

에이전트의 모든 LLM 호출은 `client` 와 `messages` 리스트를 사용합니다.

In [1]:
import json
import os
import re
import sys
from datetime import datetime
from pathlib import Path

import pytz
import yfinance as yf
from dotenv import load_dotenv
from openai import OpenAI

BASE = Path('.').resolve()
sys.path.insert(0, str(BASE))

load_dotenv("../../.env")
api_key = os.getenv("OPENAI_KEY")
client = OpenAI(api_key=api_key)

SAMPLES = BASE / 'samples'
print('✅ client 준비')

✅ client 준비


In [2]:
def ask(system_prompt, user_prompt, temperature=0.1):
    """3.3 스타일: system + user 분리 호출"""
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        temperature=temperature,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ],
    )
    return response.choices[0].message.content

print(ask('You are a helpful assistant.', '한 문장으로 자기소개해줘.', 0.2)[:80], '...')

안녕하세요, 저는 다양한 정보를 제공하고 질문에 답변하는 AI 어시스턴트입니다. ...


---
## 2. RAG 인덱스 구축 (03 노트북)

학칙 PDF → 텍스트 → 청크 → 검색용 `INDEX`

In [3]:
from pdf_summary import summarize_pdf,pdf_to_text


pdf_path = SAMPLES / '학칙.pdf'
text_path = pdf_to_text(str(pdf_path))
with open(text_path, 'r', encoding='utf-8') as f:
    rules_text = f.read()

print('학칙 글자 수:', len(rules_text))

입력받은 pdf_path = /Users/seorincho/Desktop/code/2026_AI/SK Autonomous R&D/4일차/samples/학칙.pdf
학칙 글자 수: 46965


In [4]:
def split_text(text, chunk_size=280, overlap=60):
    text = re.sub(r'\s+', ' ', text.strip())
    if len(text) <= chunk_size:
        return [text]
    chunks, start = [], 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start = max(0, end - overlap)
    return chunks

chunks = split_text(rules_text)
print('청크 수:', len(chunks))

청크 수: 212


In [5]:
INDEX = [
    {
        'chunk_id': f'RULE_C{i:03d}',
        'title': '연세대학교 대학원 학칙',
        'text': chunk,
    }
    for i, chunk in enumerate(chunks, 1)
]
print('인덱스 청크:', len(INDEX))

인덱스 청크: 212


In [20]:
def tokenize(text):
    return {t.lower() for t in re.findall(r'[가-힣a-zA-Z0-9]+', text) if len(t) > 1}

def search(query, top_k=3): #특정 pdf의 chunk별로 token set와 q의(질문의 token set) 교집합 단어들을 찾아서 유사성을 판단
    q = tokenize(query)
    scored = []
    for item in INDEX:
        score = len(q & tokenize(item['text']))
        if score > 0:
            scored.append((score, item))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [{'score': s, **it} for s, it in scored[:top_k]]

demo_q = '석사학위과정 수업연한은?'
print('----------------------------')
print(search(demo_q,top_k=2))
print('----------------------------')
for h in search(demo_q, top_k=2):
    print(f"[{h['chunk_id']}] score={h['score']}", h['text'][:90], '...')

----------------------------
[{'score': 2, 'chunk_id': 'RULE_C005', 'title': '연세대학교 대학원 학칙', 'text': '-- 8 ③ <삭제> 제2조의7(수업연한) ① 대학원 각 학위과정의 수업연한은 다음 각호와 같다. 1. 석사학위과정: 2년(4학기) 2. 박사학위과정: 2년(4학기) 3. 통합과정: 3년(6학기) ② 제1항의 규정에도 불구하고 학위수여 자격 요건을 충족한 학생은 수업연한을 1 학기 단축할 수 있다. 단, 통합과정 및 석사학위논문 대체실적을 통해 학위를 취득 하고자 하는 학생은 제외한다. 제2조의8(재학연한) ① 대학원 각 학위과정의 재학연한은 다음 각호와 같으며, 이를 초 과할 수 없다. 단, 휴'}, {'score': 1, 'chunk_id': 'RULE_C001', 'title': '연세대학교 대학원 학칙', 'text': 'Ⅰ. 대학원 학칙 / 7 Ⅰ. 대학원 학칙 제정: 1974. 05. 18. 개정(제122차): 2026. 06. 06. 제1장 총칙 제1조(목적) 대학원은 기독교 정신을 바탕으로 하여 창의적 이론과 과학적 방법을 탐 구하고, 지도적 인격을 도야하여 인류 문화 향상에 기여함을 목적으로 한다. 제2장 과정 및 정원 제2조(과정) 대학원에는 석사학위를 취득하기 위한 석사학위과정, 박사학위를 취득하기 위한 박사학위과정, 석사학위과정을 거치지 않고 박사학위를 취득하기 위하여 석사 학위과정과 박사학위과정이 '}]
----------------------------
[RULE_C005] score=2 -- 8 ③ <삭제> 제2조의7(수업연한) ① 대학원 각 학위과정의 수업연한은 다음 각호와 같다. 1. 석사학위과정: 2년(4학기) 2. 박사학위과정: 2년(4학 ...
[RULE_C001] score=1 Ⅰ. 대학원 학칙 / 7 Ⅰ. 대학원 학칙 제정: 1974. 05. 18. 개정(제122차): 2026. 06. 06. 제1장 총칙 제1조(목적) 대학원은 기독교 ...

---
## 3. Tool 함수 구현 — 하나씩 만들기

각 기능을 Python 함수로 만든 뒤, 다음 섹션에서 **tool 스키마**로 등록합니다.

### 3-1. 학칙 RAG 검색 (03)

In [8]:
def search_university_rules(query: str, top_k: int = 3) -> str:
    """학칙 PDF 인덱스에서 관련 청크를 검색합니다."""
    hits = search(query, top_k=top_k)
    if not hits:
        return json.dumps({'query': query, 'chunks': [], 'message': '검색 결과 없음'}, ensure_ascii=False)
    return json.dumps({
        'query': query,
        'chunks': [
            {'chunk_id': h['chunk_id'], 'score': h['score'], 'text': h['text']}
            for h in hits
        ],
    }, ensure_ascii=False)

print(search_university_rules('석사 수업연한')[:200], '...')

{"query": "석사 수업연한", "chunks": [{"chunk_id": "RULE_C004", "score": 2, "text": "연계하는 학부 -대학원 연계과정을 둔다. 연계과정 운영에 관한 사항은 따로 정한다. 제2조의6(계약학과) ① 대학원에 두는 학위과정으로 국가, 지방자치단체 또는 산업체 등과의 계약에 의한 학과를 둘 수 있으며, 이의  ...


### 3-2. 도시 현재 시각 (3.4 — pytz)

In [9]:
CITY_TZ = {
    '서울': 'Asia/Seoul', 'seoul': 'Asia/Seoul',
    '뉴욕': 'America/New_York', 'new york': 'America/New_York',
    '도쿄': 'Asia/Tokyo', 'tokyo': 'Asia/Tokyo',
    '런던': 'Europe/London', 'london': 'Europe/London',
}

def get_city_time_tz(city: str) -> str:
    key = city.strip().lower()
    tz_name = CITY_TZ.get(key)
    if not tz_name:
        return json.dumps({'error': f'지원하지 않는 도시: {city}'}, ensure_ascii=False)
    now = datetime.now(pytz.timezone(tz_name)).strftime('%Y-%m-%d %H:%M:%S')
    return json.dumps({'city': city, 'timezone': tz_name, 'current_time': now}, ensure_ascii=False)

print(get_city_time_tz('서울'))

{"city": "서울", "timezone": "Asia/Seoul", "current_time": "2026-06-25 16:52:02"}


### 3-3. 미국 주식 가격 (3.4 — yfinance)

In [10]:
def get_us_stock_price(ticker: str) -> str:
    symbol = ticker.strip().upper()
    try:
        hist = yf.Ticker(symbol).history(period='2d')
        if hist.empty:
            return json.dumps({'error': f'{symbol} 데이터가 없습니다.'}, ensure_ascii=False)
        latest = hist.iloc[-1]
        return json.dumps({
            'ticker': symbol,
            'open': round(float(latest['Open']), 2),
            'close': round(float(latest['Close']), 2),
            'currency': 'USD',
            'source': 'yfinance',
        }, ensure_ascii=False)
    except Exception as e:
        return json.dumps({'error': str(e)}, ensure_ascii=False)

print(get_us_stock_price('AAPL'))

{"ticker": "AAPL", "open": 295.36, "close": 293.08, "currency": "USD", "source": "yfinance"}


### 3-4. PDF 문서 요약 (01 — pdf_summary)

In [11]:
from pdf_summary import summarize_pdf,pdf_to_text

def summarize_pdf_document(pdf_name: str) -> str:
    """samples 폴더의 PDF를 요약합니다. 예: Language_Models.pdf"""
    pdf_path = SAMPLES / pdf_name
    if not pdf_path.exists():
        return json.dumps({'error': f'파일 없음: {pdf_name}'}, ensure_ascii=False)
    stem = pdf_path.stem
    out_path = SAMPLES / f'{stem}_summary.txt'
    summary = summarize_pdf(str(pdf_path))
    return json.dumps({'pdf': pdf_name, 'summary': summary}, ensure_ascii=False)

# 데모 (LLM 호출 포함 — 시간이 걸릴 수 있음)
print(json.loads(summarize_pdf_document('Language_Models.pdf'))['summary'][:300], '...')

입력받은 pdf_path = /Users/seorincho/Desktop/code/2026_AI/SK Autonomous R&D/4일차/samples/Language_Models.pdf
.env loaded: /Users/seorincho/Desktop/code/2026_AI/.env
# Language Models are Few-Shot Learners

## 저자의 문제 인식 및 주장
저자들은 최근 자연어 처리(NLP) 분야에서 대규모 텍스트 코퍼스를 기반으로 한 사전 학습과 특정 작업에 대한 미세 조정이 많은 성과를 거두었다고 주장한다. 그러나 이러한 방법은 여전히 수천 또는 수만 개의 작업별 데이터셋을 필요로 하며, 이는 새로운 언어 작업을 수행하는 데 있어 인간이 몇 가지 예시만으로도 가능하다는 점과 대조된다. 저자들은 언어 모델의 규모를 확장함으로써 작업에 구애받지 않는 몇 가지 샷 성능이 크게 향상된 ...


---
## 4. Tool 스키마 등록 + 통합 (3.4)

함수 4개를 **하나의 `AGENT_TOOLS` 리스트**와 **`TOOL_FUNCTIONS` 딕셔너리**로 묶습니다.
LLM은 질문 내용에 맞는 tool을 **스스로 고릅니다**.

In [12]:
AGENT_TOOLS = [
    {
        'type': 'function',
        'function': {
            'name': 'get_city_time_tz',
            'description': '도시의 시간대를 반영해 현재 시각을 반환합니다. 서울·뉴욕·도쿄·런던 등.',
            'parameters': {
                'type': 'object',
                'properties': {'city': {'type': 'string', 'description': '도시 이름 (예: 서울, 뉴욕)'}},
                'required': ['city'],
            },
        },
    },
    {
        'type': 'function',
        'function': {
            'name': 'get_us_stock_price',
            'description': '미국 주식 티커의 최근 시가·종가를 조회합니다. 예: AAPL, TSLA, NVDA.',
            'parameters': {
                'type': 'object',
                'properties': {'ticker': {'type': 'string', 'description': '주식 티커 심볼'}},
                'required': ['ticker'],
            },
        },
    },
    {
        'type': 'function',
        'function': {
            'name': 'summarize_pdf_document',
            'description': 'samples 폴더에 있는 PDF 문서를 읽고 요약합니다.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'pdf_name': {'type': 'string', 'description': 'PDF 파일명 (예: Language_Models.pdf)'},
                },
                'required': ['pdf_name'],
            },
        },
    },
    {
        'type': 'function',
        'function': {
            'name': 'search_university_rules',
            'description': '연세대학교 대학원 학칙에서 질문과 관련된 조항을 검색합니다. 학칙·규정·학점·휴학·논문 등 사실 질문에 사용.',
            'parameters': {
                'type': 'object',
                'properties': {
                    'query': {'type': 'string', 'description': '검색할 질문 또는 키워드'},
                    'top_k': {'type': 'integer', 'description': '가져올 청크 개수 (기본 3)'},
                },
                'required': ['query'],
            },
        },
    },
]

TOOL_FUNCTIONS = {
    'get_city_time_tz': get_city_time_tz,
    'get_us_stock_price': get_us_stock_price,
    'summarize_pdf_document': summarize_pdf_document,
    'search_university_rules': search_university_rules,
}

print('등록된 tool:', list(TOOL_FUNCTIONS.keys()))

등록된 tool: ['get_city_time_tz', 'get_us_stock_price', 'summarize_pdf_document', 'search_university_rules']


---
## 5. 통합 Agent System Prompt (3.3)

LLM이 **어떤 tool을 쓸지** 판단하도록 역할·라우팅 규칙을 정의합니다.

In [13]:
AGENT_SYSTEM = '''
너는 다목적 업무 도우미 Agent입니다. 제공된 도구를 적절히 사용해 답변하세요.

도구 선택 가이드:
- 현재 시각·시간대 → get_city_time_tz
- 미국 주식 가격 → get_us_stock_price
- samples 폴더 PDF 요약 → summarize_pdf_document (학칙 질문에는 사용하지 마세요)
- 연세대 대학원 학칙·규정 사실 질문 → search_university_rules

규칙:
- 실시간·문서·학칙 정보가 필요하면 반드시 도구를 먼저 호출하세요.
- 도구 결과에 없는 내용은 추측하지 마세요.
- 답변은 한국어, bullet 3개 이내로 간결하게.
- 학칙 답변 시 근거 chunk_id 를 마지막 bullet에 적으세요. (예: 근거: RULE_C039)
'''.strip()

print(AGENT_SYSTEM)

너는 다목적 업무 도우미 Agent입니다. 제공된 도구를 적절히 사용해 답변하세요.

도구 선택 가이드:
- 현재 시각·시간대 → get_city_time_tz
- 미국 주식 가격 → get_us_stock_price
- samples 폴더 PDF 요약 → summarize_pdf_document (학칙 질문에는 사용하지 마세요)
- 연세대 대학원 학칙·규정 사실 질문 → search_university_rules

규칙:
- 실시간·문서·학칙 정보가 필요하면 반드시 도구를 먼저 호출하세요.
- 도구 결과에 없는 내용은 추측하지 마세요.
- 답변은 한국어, bullet 3개 이내로 간결하게.
- 학칙 답변 시 근거 chunk_id 를 마지막 bullet에 적으세요. (예: 근거: RULE_C039)


---
## 6. Tool Calling 수동 1회 (3.4)

에이전트 루프 3단계를 직접 따라갑니다.

1. LLM → tool_calls 요청
2. Python 함수 실행 → `role: tool` 로 결과 전달
3. LLM → 최종 답변

In [14]:
user_question = '지금 서울은 몇 시야?'

messages = [
    {'role': 'system', 'content': AGENT_SYSTEM},
    {'role': 'user', 'content': user_question},
]

first = client.chat.completions.create(
    model='gpt-4o-mini', temperature=0.1, messages=messages, tools=AGENT_TOOLS,
)
msg = first.choices[0].message
print('tool_calls:', [tc.function.name for tc in (msg.tool_calls or [])])

tool_calls: ['get_city_time_tz']


In [15]:
messages.append(msg)

for tc in msg.tool_calls:
    fn_name = tc.function.name
    fn_args = json.loads(tc.function.arguments or '{}')
    result = TOOL_FUNCTIONS[fn_name](**fn_args)
    print(f'  {fn_name}({fn_args})')
    messages.append({'role': 'tool', 'tool_call_id': tc.id, 'content': result})

final = client.chat.completions.create(
    model='gpt-4o-mini', temperature=0.1, messages=messages, tools=AGENT_TOOLS,
)
print('\n=== 최종 답변 ===')
print(final.choices[0].message.content)

  get_city_time_tz({'city': '서울'})

=== 최종 답변 ===
현재 서울은 2026년 6월 25일 16시 52분입니다.


---
## 7. `run_agent_once()` — 통합 Agent 실행 (3.4)

In [16]:
def run_agent_once(user_question, tools, tool_functions, system_msg):
    messages = [
        {'role': 'system', 'content': system_msg},
        {'role': 'user', 'content': user_question},
    ]
    first = client.chat.completions.create(
        model='gpt-4o-mini', temperature=0.1, messages=messages, tools=tools,
    )
    msg = first.choices[0].message
    if not msg.tool_calls:
        return msg.content
    messages.append(msg)
    for tc in msg.tool_calls:
        fn = tc.function.name
        args = json.loads(tc.function.arguments or '{}')
        result = tool_functions[fn](**args)
        print(f'  → {fn}({args})')
        messages.append({'role': 'tool', 'tool_call_id': tc.id, 'content': result})
    final = client.chat.completions.create(
        model='gpt-4o-mini', temperature=0.1, messages=messages, tools=tools,
    )
    return final.choices[0].message.content

In [17]:
questions = [
    '지금 서울 시간 알려줘',
    '애플(AAPL) 주가 알려줘',
    '석사학위과정 수업연한은?',
    '서울 시간이랑 테슬라(TSLA) 주가 같이 알려줘',
]

for q in questions:
    print('=' * 60)
    print('Q:', q)
    print(run_agent_once(q, AGENT_TOOLS, TOOL_FUNCTIONS, AGENT_SYSTEM))
    print()

Q: 지금 서울 시간 알려줘
  → get_city_time_tz({'city': '서울'})
현재 서울 시간은 2026년 6월 25일 16시 52분 38초입니다.

Q: 애플(AAPL) 주가 알려줘
  → get_us_stock_price({'ticker': 'AAPL'})
- 애플(AAPL) 주가는 현재 293.08 USD입니다.
- 개장가는 295.36 USD였습니다.

Q: 석사학위과정 수업연한은?
  → search_university_rules({'query': '석사학위과정 수업연한'})
- 석사학위과정의 수업연한은 2년(4학기)입니다.
- 학위수여 자격 요건을 충족한 경우, 수업연한을 1학기 단축할 수 있습니다. 단, 특정 조건이 있습니다.
- 근거: RULE_C005

Q: 서울 시간이랑 테슬라(TSLA) 주가 같이 알려줘
  → get_city_time_tz({'city': '서울'})
  → get_us_stock_price({'ticker': 'TSLA'})
- 현재 서울 시간: 2026년 6월 25일 16:52:46
- 테슬라(TSLA) 주가: 시가 380.08 USD, 종가 375.53 USD



In [18]:
# PDF 요약 (LLM 호출 2회 — 시간 소요)
q = 'Language_Models.pdf 문서 요약해줘'
print('Q:', q)
print(run_agent_once(q, AGENT_TOOLS, TOOL_FUNCTIONS, AGENT_SYSTEM))

Q: Language_Models.pdf 문서 요약해줘
입력받은 pdf_path = /Users/seorincho/Desktop/code/2026_AI/SK Autonomous R&D/4일차/samples/Language_Models.pdf
.env loaded: /Users/seorincho/Desktop/code/2026_AI/.env
  → summarize_pdf_document({'pdf_name': 'Language_Models.pdf'})
- 저자들은 대규모 텍스트 코퍼스를 사전 훈련한 후 특정 작업에 대해 미세 조정하는 방법이 성과를 거두었지만, 여전히 많은 예제가 필요하다고 주장합니다.
- GPT-3 모델은 1750억 개의 매개변수를 가지고 있으며, 미세 조정 없이도 다양한 NLP 작업에서 강력한 성능을 발휘합니다.
- 그러나 GPT-3는 일부 데이터셋에서 어려움을 겪고 있으며, 생성한 뉴스 기사가 인간 작성자와 구별하기 어려운 경우도 발견되었습니다.


---
## 8. 멀티턴 (3.2) — 이어서 질문하기

한 대화 안에서 tool 종류가 바뀌는지 확인합니다.

In [19]:
chat_messages = [
    {'role': 'system', 'content': AGENT_SYSTEM},
    {'role': 'user', 'content': '박사학위과정 재학연한은?'},
]

r1 = client.chat.completions.create(
    model='gpt-4o-mini', temperature=0.1, messages=chat_messages, tools=AGENT_TOOLS,
)
m1 = r1.choices[0].message
chat_messages.append(m1)

if m1.tool_calls:
    for tc in m1.tool_calls:
        args = json.loads(tc.function.arguments or '{}')
        chat_messages.append({
            'role': 'tool', 'tool_call_id': tc.id,
            'content': TOOL_FUNCTIONS[tc.function.name](**args),
        })
    r1b = client.chat.completions.create(
        model='gpt-4o-mini', temperature=0.1, messages=chat_messages, tools=AGENT_TOOLS,
    )
    a1 = r1b.choices[0].message.content
    chat_messages.append({'role': 'assistant', 'content': a1})
else:
    a1 = m1.content

print('1차 (학칙):', a1)

chat_messages.append({'role': 'user', 'content': '내가 22년도에 입학했는데 그럼 지금으로부터 몇년 남은거지?'})
r2 = client.chat.completions.create(
    model='gpt-4o-mini', temperature=0.1, messages=chat_messages, tools=AGENT_TOOLS,
)
m2 = r2.choices[0].message
chat_messages.append(m2)

if m2.tool_calls:
    for tc in m2.tool_calls:
        args = json.loads(tc.function.arguments or '{}')
        chat_messages.append({
            'role': 'tool', 'tool_call_id': tc.id,
            'content': TOOL_FUNCTIONS[tc.function.name](**args),
        })
    r2b = client.chat.completions.create(
        model='gpt-4o-mini', temperature=0.1, messages=chat_messages, tools=AGENT_TOOLS,
    )
    a2 = r2b.choices[0].message.content
else:
    a2 = m2.content

print('\n2차 (시간):', a2)

1차 (학칙): - 박사학위과정의 재학연한은 7년(14학기)입니다.
- 휴학 및 제적 기간은 재학연한에 포함되지 않습니다.
- 근거: RULE_C006

2차 (시간): - 2022년에 입학하셨다면, 2026년까지 총 4년의 재학연한이 남았습니다.
- 현재는 2026년 6월 25일로, 박사학위과정의 재학연한이 7년이므로 1년 6개월이 남았습니다.
